## Find biggest angle in transformed prediction data

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from math import pi, degrees

def analyze_rotations(pred_dir):
    """Analyze rotation values in prediction files."""
    all_rotations = []
    rotation_by_class = {1: [], 2: [], 3: []} # Car, Pedestrian, Cyclist

    pred_files = glob.glob(os.path.join(pred_dir, '*.txt'))
    print(f"Processing {len(pred_files)} files....")

    for pred_file in pred_files:
        with open(pred_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 6:
                    try:
                        class_id = int(float(parts[0]))
                        rotation = float(parts[5])
                        all_rotations.append(rotation)

                        if class_id in rotation_by_class:
                            rotation_by_class[class_id].append(rotation)
                    except (ValueError, IndexError) as e:
                        print(f"Error processing line in {pred_file}: {line.strip()}")
                        print(f"Error details: {e}")

    print(f"Total predictions analyzed: {len(all_rotations)}")

    all_rotations_deg = [degrees(r) for r in all_rotations]

    print("\Rotation Statistics (in °)")
    print(f"Max: {max(all_rotations_deg):.2f}°")
    print(f"Min: {min(all_rotations_deg):.2f}°")
    print(f"Mean: {np.mean(all_rotations_deg):.2f}°")
    
    # Check if rotations are in the expected range [0, pi/2]
    in_range = sum(1 for r in all_rotations if 0 <= r <= pi/2)
    outside_range = len(all_rotations) - in_range
    print(f"\nRotations in range [0, pi/2]: {in_range} ({in_range/len(all_rotations)*100:.2f}%)")
    print(f"Rotations outside range: {outside_range} ({outside_range/len(all_rotations)*100:.2f}%)")
    
    # Distribution by class
    for class_id, rotations in rotation_by_class.items():
        if rotations:
            rotations_deg = [degrees(r) for r in rotations]
            class_name = {1: "Car", 2: "Pedestrian", 3: "Cyclist"}.get(class_id, f"Class {class_id}")
            print(f"\n{class_name} Rotations:")
            print(f"  Count: {len(rotations)}")
            print(f"  Min: {min(rotations_deg):.2f}°")
            print(f"  Max: {max(rotations_deg):.2f}°")
    
    # Plot histogram
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.hist(all_rotations_deg, bins=36, alpha=0.7)
    plt.title('Rotation Distribution (degrees)')
    plt.xlabel('Rotation (degrees)')
    plt.ylabel('Frequency')
    plt.axvline(x=0, color='r', linestyle='--', alpha=0.5)
    plt.axvline(x=90, color='r', linestyle='--', alpha=0.5)
    
    plt.subplot(1, 2, 2)
    plt.hist(all_rotations, bins=36, alpha=0.7)
    plt.title('Rotation Distribution (radians)')
    plt.xlabel('Rotation (radians)')
    plt.ylabel('Frequency')
    plt.axvline(x=0, color='r', linestyle='--', alpha=0.5)
    plt.axvline(x=pi/2, color='r', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    
    # Print a few example rotations
    print("\nSample rotations (radians → degrees):")
    sample_size = min(10, len(all_rotations))
    for i in range(sample_size):
        print(f"  {all_rotations[i]:.4f} rad → {degrees(all_rotations[i]):.2f}°")

# Erweiterte Funktion für beide Analysen
def analyze_all_rotations():
    """Analyze rotation values in both BEV prediction and LiDAR files."""
    
    # 1. Analyze BEV Predictions
    bev_pred_dir = "/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/all_bev_preds/val_trt_fp32/labels"
    if not os.path.exists(bev_pred_dir):
        print(f"BEV directory '{bev_pred_dir}' not found.")
    else:
        print("="*50)
        print("ANALYZING BEV PREDICTIONS")
        print("="*50)
        analyze_rotations(bev_pred_dir)
    
    # 2. Analyze LiDAR Predictions
    lidar_pred_dir = "/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/pred_bev_to_lidar_fp32"
    if not os.path.exists(lidar_pred_dir):
        print(f"LiDAR directory '{lidar_pred_dir}' not found.")
    else:
        print("\n\n" + "="*50)
        print("ANALYZING LIDAR PREDICTIONS")
        print("="*50)
        
        # Special function for LiDAR format
        all_rotations = []
        rotation_by_class = {"Car": [], "Pedestrian": [], "Cyclist": []}
        
        pred_files = glob.glob(os.path.join(lidar_pred_dir, "*.txt"))
        print(f"Processing {len(pred_files)} LiDAR files...")
        
        for pred_file in pred_files:
            with open(pred_file, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 12:  # LiDAR format has more fields
                        try:
                            obj_type = parts[0]
                            rotation = float(parts[11])  # rotation_z is at index 11 in LiDAR format
                            all_rotations.append(rotation)
                            
                            if obj_type in rotation_by_class:
                                rotation_by_class[obj_type].append(rotation)
                        except (ValueError, IndexError) as e:
                            print(f"Error processing line in {pred_file}: {line.strip()}")
                            print(f"Error details: {e}")
        
        print(f"Total LiDAR predictions analyzed: {len(all_rotations)}")
        
        all_rotations_deg = [degrees(r) for r in all_rotations]
        
        print("\nLiDAR Rotation Statistics (in °)")
        print(f"Min: {min(all_rotations_deg):.2f}°")
        print(f"Max: {max(all_rotations_deg):.2f}°")
        print(f"Mean: {np.mean(all_rotations_deg):.2f}°")
        
        # Check if rotations are in the expected range [0, pi/2]
        in_range = sum(1 for r in all_rotations if 0 <= r <= pi/2)
        outside_range = len(all_rotations) - in_range
        print(f"\nLiDAR Rotations in range [0, pi/2]: {in_range} ({in_range/len(all_rotations)*100:.2f}%)")
        print(f"LiDAR Rotations outside range: {outside_range} ({outside_range/len(all_rotations)*100:.2f}%)")
        
        # Check distribution in other ranges
        ranges = [(-pi, -pi/2), (-pi/2, 0), (0, pi/2), (pi/2, pi), (pi, 3*pi/2)]
        range_labels = ["[-π, -π/2]", "[-π/2, 0]", "[0, π/2]", "[π/2, π]", "[π, 3π/2]"]
        
        for i, (low, high) in enumerate(ranges):
            count = sum(1 for r in all_rotations if low <= r < high)
            print(f"LiDAR Rotations in {range_labels[i]}: {count} ({count/len(all_rotations)*100:.2f}%)")
        
        # Distribution by class
        for obj_type, rotations in rotation_by_class.items():
            if rotations:
                rotations_deg = [degrees(r) for r in rotations]
                print(f"\n{obj_type} Rotations:")
                print(f"  Count: {len(rotations)}")
                print(f"  Min: {min(rotations_deg):.2f}°")
                print(f"  Max: {max(rotations_deg):.2f}°")
                print(f"  Mean: {np.mean(rotations_deg):.2f}°")
        
        # Plot histogram
        plt.figure(figsize=(14, 8))
        
        plt.subplot(2, 2, 1)
        plt.hist(all_rotations_deg, bins=36, alpha=0.7, range=(-180, 180))
        plt.title('LiDAR Rotation Distribution (degrees)')
        plt.xlabel('Rotation (degrees)')
        plt.ylabel('Frequency')
        plt.axvline(x=0, color='r', linestyle='--', alpha=0.5)
        plt.axvline(x=90, color='r', linestyle='--', alpha=0.5)
        plt.axvline(x=-90, color='g', linestyle='--', alpha=0.5)
        
        plt.subplot(2, 2, 2)
        plt.hist(all_rotations, bins=36, alpha=0.7, range=(-pi, pi))
        plt.title('LiDAR Rotation Distribution (radians)')
        plt.xlabel('Rotation (radians)')
        plt.ylabel('Frequency')
        plt.axvline(x=0, color='r', linestyle='--', alpha=0.5)
        plt.axvline(x=pi/2, color='r', linestyle='--', alpha=0.5)
        plt.axvline(x=-pi/2, color='g', linestyle='--', alpha=0.5)
        
        # Class-specific distributions
        plt.subplot(2, 2, 3)
        for obj_type, rotations in rotation_by_class.items():
            if rotations:
                rotations_deg = [degrees(r) for r in rotations]
                plt.hist(rotations_deg, bins=18, alpha=0.5, label=obj_type)
        plt.title('Rotation by Object Type (degrees)')
        plt.xlabel('Rotation (degrees)')
        plt.ylabel('Frequency')
        plt.legend()
        
        plt.tight_layout()
        
        # Print a few example rotations
        print("\nSample LiDAR rotations (radians → degrees):")
        sample_size = min(10, len(all_rotations))
        for i in range(sample_size):
            print(f"  {all_rotations[i]:.4f} rad → {degrees(all_rotations[i]):.2f}°")
            


if __name__ == "__main__":
    pred_dir = "/home/heizung1/view-of-delft-dataset/vod/label_transformation/predictions/all_bev_preds/val_trt_fp32/labels"

    if not os.path.exists(pred_dir):
        print(f"Directory '{pred_dir}' not found.")
        exit(1)
    
    #analyze_rotations(pred_dir)
    # Ausführen der Analyse
    analyze_all_rotations()


## gt_bev2lidar_frame test
#### current topic: rotation calculation

In [ ]:
import numpy as np 
import pickle
from math import atan2, pi, sqrt
from vod.label_transformation.utils.utils import normalize_angle, bev_to_pixel_coords, pixel_to_world_coords
from vod.label_transformation.utils.utils import extract_gt_for_lidar_idx, save_transf_lidar_labels

class BEVtoLiDARConverter:
    def __init__(self, image_size=(640, 640), cell_size=0.1):
        self.image_width, self.image_height = image_size
        self.cell_size = cell_size

    def bev_to_lidar_label(self, bev_label, gt_match=None, gt_rotation=None):
        """
        Convert a BEV label to a LiDAR label.
        
        Args:
            bev_label: List containing [class_id, x1, y1, x2, y2, x3, y3, x4, y4, bbox, truncation, occlusion]
            gt_match: ground truth match dictionary (optional)
        Returns:
        lidar_label: Dictionary containing the LiDAR label fields    
        """
        class_id = int(bev_label[0])
        x1, y1, x2, y2, x3, y3, x4, y4 = bev_label[1:9]
        #bbox = bev_label[9:13]
        truncation = bev_label[13]
        occlusion = bev_label[14]

        # Reorder points from YOLO to standard clockwise
        """
        YOLO-Format                     Default
       x4,y4 --- x1,y1                x2,y2 --- x3,y3
        |           |                  |           |
        |           |       →          |           |
        |           |                  |           |
       x3,y3 --- x2,y2                x1,y1 --- x4,y4
        """
        x1, x2, x3, x4 = x3, x4, x1, x2
        y1, y2, y3, y4 = y3, y4, y1, y2

        # Convert from normalized pixel coords to pixel coords
        pixel_coords = bev_to_pixel_coords(
            [(x1, y1), (x2, y2), (x3, y3), (x4, y4)],
            self.image_width,
            self.image_height
        )

        # Convert from pixel coords to world coords
        world_coords = pixel_to_world_coords(
            pixel_coords,
            self.image_width,
            self.image_height,
            self.cell_size
        )

        # Calculate center, dimensions, and rotation from BEV data
        center =  np.mean(world_coords, axis=0)
        edge = world_coords[1] - world_coords[0]
        length = np.linalg.norm(edge)
        width = np.linalg.norm(world_coords[2] - world_coords[1])
        
        # Calculate the rotation directly from the world coordinates of the bounding box
        heading_lidar = atan2(edge[1], edge[0]) # y, x, directly provides the correct angle in the LiDAR CW system
        heading_lidar = normalize_angle(heading_lidar, gt_rotation) # [-pi...pi]
        #print(f"(Calc.) x,y: {center[0], center[1]}")
        #print(f"(Calc.) rot_z: {heading_lidar}")
        print(f"{heading_lidar:.3f}")

        # TODO: Test approach
        edge_v = world_coords[1] - world_coords[0]  # 0->1 (vertical)
        edge_h = world_coords[2] - world_coords[1]  #1->2 (horizontal)

        if class_id == 1 or class_id == 3: 
            if np.linalg.norm(edge_h) > np.linalg.norm(edge_v):
                orientation_vector = edge_h
                length = np.linalg.norm(edge_h)
                width = np.linalg.norm(edge_v)
            else:
                orientation_vector = edge_v
                length = np.linalg.norm(edge_v)
                width = np.linalg.norm(edge_h)
        else:
            orientation_vector = edge_v
            length = np.linalg.norm(edge_v)
            width = np.linalg.norm(edge_h)

        test = atan2(orientation_vector[1], orientation_vector[0])
        #print(f"{test:.3f}")
 
        # Don't subtract the class-specific offsets applied before training, it would distort the metrics
        #if class_id == 1:  # Car
        #    length = max(0, length - 0.4)
        #    width = max(0, width - 0.4)
        #elif class_id in [2, 3]:  # Pedestrian/Cyclist
        #    length = max(0, length - 0.3)
        #    width = max(0, width - 0.3)
        #print(f"(Calc.) w,l: {width, length}")

        # Extract alpha, score, height, z from gt_match in .pkl file
        alpha = gt_match.get('alpha', -10) if gt_match else -10
        score = gt_match.get('score', 0.0) if gt_match else 0.0
        height = gt_match['3Dbox'][5] if gt_match else -1
        z = gt_match['3Dbox'][2] if gt_match else -1000
  
        # Map class_id to type
        class_map = {1: "Car", 2: "Pedestrian", 3: "Cyclist"}
        obj_type = class_map.get(class_id, "DontCare")

        #print(f"(.pkl) h, z: {height, z}")

        # Create LiDAR label
        lidar_label = {
            "type": str(obj_type),
            "truncated": float(truncation),
            "occluded": int(occlusion),
            "alpha": alpha, # from gt data
            #"bbox": bbox,  # from gt data
            "bbox": [-1, -1, -1, -1], # dummy values, calc. later
            "dimensions": [height, width, length], #  h, w, l
            "location": [center[0], center[1], z], # x, y, z
            "rotation_z": heading_lidar,
            "score": float(score) # from gt data
        }

        return lidar_label
    
    def match_bev_to_gt(self, bev_label, gt_info):
        """
        Match a BEV label to the closest ground truth box without creating a full LiDAR label.
        
        Args:
            bev_label: List containing BEV label information
            gt_info: List of ground truth boxes
            
        Returns:
            best_match: Closest ground truth box dictionary or None
        """
        class_id = int(bev_label[0])
        class_map = {1: 'Car', 2: 'Pedestrian', 3: 'Cyclist'}
        
        # Extract BEV coordinates without creating a full LiDAR label
        x1, y1, x2, y2, x3, y3, x4, y4 = bev_label[1:9]
        
        # Reorder points from YOLO to standard clockwise
        x1, x2, x3, x4 = x3, x4, x1, x2
        y1, y2, y3, y4 = y3, y4, y1, y2

        # Convert to pixel coordinates
        pixel_coords = bev_to_pixel_coords(
            [(x1, y1), (x2, y2), (x3, y3), (x4, y4)],
            self.image_width,
            self.image_height
        )

        # Convert to world coordinates
        world_coords = pixel_to_world_coords(
            pixel_coords,
            self.image_width,
            self.image_height,
            self.cell_size
        )

        # Calculate center and dimensions without storing them as a label
        center = np.mean(world_coords, axis=0)
        edge = world_coords[1] - world_coords[0]
        length = np.linalg.norm(edge)
        width = np.linalg.norm(world_coords[2] - world_coords[1])
        
        # Calculate rotation angle
        heading = atan2(edge[1], edge[0])
        
        best_match = None
        min_score = float('inf')

        for gt in gt_info:
            # Class filter
            if gt['name'] != class_map.get(class_id, 'DontCare'):
                continue

            gt_box3d = gt['3Dbox']

            # Position difference (Euclidean distance)
            pos_diff = sqrt((center[0] - gt_box3d[0])**2 + 
                            (center[1] - gt_box3d[1])**2)

            # Dimension difference
            dim_diff = min(
                abs(length - gt_box3d[3]) + abs(width - gt_box3d[4]),
                abs(length - gt_box3d[4]) + abs(width - gt_box3d[3])
            )

            # Rotation difference
            rot_diff = abs(heading - gt_box3d[6])
            rot_diff = min(rot_diff, 2 * pi - rot_diff)

            # Combined score (weighted)
            score = 0.6 * pos_diff + 0.2 * dim_diff + 0.2 * rot_diff

            if score < min_score:
                min_score = score
                best_match = gt
        
        return best_match if min_score < 2.0 else None
    
if __name__ == "__main__":
    import os

    single_file_mode = False

    if single_file_mode:
        # Path to file with bev labels
        bev_label_file = "kitti_gt_annos_2/all_bev_gt_annos_2/bev_val_000076.txt"

        # Check if the file exists
        if not os.path.exists(bev_label_file):
            print(f"BEV label file '{bev_label_file}' not found.")
            exit(1)

        # Load BEV labels from the file
        bev_labels = []
        with open(bev_label_file, "r") as f:
            for line in f:
                bev_labels.append([float(x) for x in line.strip().split()])

        # Load ground truth data
        with open("validation_pickle/kitti_val_dataset.pkl", "rb") as f:
            gt_data = pickle.load(f)

        # Initialise converter
        converter = BEVtoLiDARConverter()

        # Convert BEV labels into LiDAR labels and output them
        for i, bev_label in enumerate(bev_labels):
            lidar_idx = bev_label_file.split('_')[-1].split('.')[0]
            gt_boxes = extract_gt_for_lidar_idx(gt_data, lidar_idx)

            gt_match = converter.match_bev_to_gt(bev_label, gt_boxes)
            lidar_label = converter.bev_to_lidar_label(bev_label, gt_match, gt_match['3Dbox'][6])

            print(f"LiDAR label {i + 1}:")
            for key, value in lidar_label.items():
                print(f"  {key}: {value}")
            print("\n")

            #save_transf_lidar_labels("kitti_gt_annos/gt_bev_to_lidar_labels", lidar_idx, [lidar_label])
    else:
        import glob
        from progress.bar import IncrementalBar
    
        bev_label_dir = "kitti_gt_annos_2/all_bev_gt_annos_2"
        output_dir = "kitti_gt_annos_2/test"

        if not os.path.exists(bev_label_dir):
            print(f"Directory '{bev_label_dir}' not found.")
            exit(1)
        
        with open("validation_pickle/kitti_val_dataset.pkl", "rb") as f:
            gt_data_pkl = pickle.load(f)

        converter = BEVtoLiDARConverter()

        bev_label_files = glob.glob(os.path.join(bev_label_dir, "*.txt"))

        bar = IncrementalBar('Processing', max=len(bev_label_files), 
                             suffix='%(percent).1f%% - Estimated time: %(eta)ds')
    
        for bev_label_file in bev_label_files:
            bev_labels = []
            with open(bev_label_file, "r") as f:
                for line in f:
                    bev_labels.append([float(x) for x in line.strip().split()])

            lidar_idx = os.path.basename(bev_label_file).split('_')[-1].split('.')[0]
            gt_info = extract_gt_for_lidar_idx(gt_data_pkl, lidar_idx)

            lidar_labels = []
            for bev_label in bev_labels:
                gt_match = converter.match_bev_to_gt(bev_label, gt_info)
                lidar_label = converter.bev_to_lidar_label(bev_label, gt_match, gt_match['3Dbox'][6])
                lidar_labels.append(lidar_label)

            #save_transf_lidar_labels(output_dir, lidar_idx, lidar_labels)

            bar.next()

        bar.finish()


# TODO: test if offset is really needed or hurts IoU during evaluation
                                                      

## gt_lidar2camera_frame
#### current topic: should I recalculate the 2D Boxes or take them from gt

In [ ]:
import numpy as np
import pickle
from pathlib import Path
import copy
from vod.label_transformation.utils.utils import save_transf_camera_labels, cart_to_hom, boxes3d_to_corners3d_kitti_camera

class LiDARtoCameraConverter:
    def __init__(self):
        """Initialize converter with calibration data from dataset"""
        self.dataset = None
        self.calib_data = {}
        self.P2 = None
        self.R0 = None
        self.V2C = None
        self.image_shape = None


    def load_calib_from_pkl(self, dataset_path):
        """Load dataset to get calibration data"""
        with open(dataset_path, 'rb') as f:
            self.dataset = pickle.load(f)
            
        # Extract calibration data for each frame
        for frame in self.dataset:
            if 'point_cloud' in frame and 'calib' in frame:
                lidar_idx = frame['point_cloud']['lidar_idx']
                calib = frame['calib']

                if 'image' in frame:
                    image_shape = frame['image']['image_shape']

                self.calib_data[lidar_idx] = {
                    'P2': calib['P2'][:3],  # 3 x 4
                    'R0': calib['R0_rect'][:3, :3],  # 3 x 3
                    'Tr_velo2cam': calib['Tr_velo_to_cam'][:3],  # 3 x 4
                    'image_shape': image_shape # [height, width]    
                }
    

    def get_calib_for_frame(self, lidar_idx):
        """Get calibration data for specific frame"""
        if lidar_idx not in self.calib_data:
            raise ValueError(f"No calibration data found for frame {lidar_idx}")
        
        calib = self.calib_data[lidar_idx]
        self.P2 = calib['P2']
        self.R0 = calib['R0']
        self.V2C = calib['Tr_velo2cam']
        self.image_shape = calib['image_shape']
        return calib
    

    def lidar_to_rect(self, pts_lidar):
        """Convert points from LiDAR to camera rect coordinates
        Args:
            pts_lidar: (N, 3)
        Returns:
            pts_rect: (N, 3)    
        """
        pts_lidar_hom = cart_to_hom(pts_lidar)
        pts_rect = np.dot(pts_lidar_hom, np.dot(self.V2C.T, self.R0.T))
        return pts_rect
    

    def rect_to_img(self, pts_rect):
        """
        :param pts_rect: (N, 3)
        :return pts_img: (N, 2)
        """
        pts_rect_hom = cart_to_hom(pts_rect)
        pts_2d_hom = np.dot(pts_rect_hom, self.P2.T)
        pts_img = (pts_2d_hom[:, 0:2].T / pts_rect_hom[:, 2]).T  # (N, 2)
        pts_rect_depth = pts_2d_hom[:, 2] - self.P2.T[3, 2]  # depth in rect camera coord

        return pts_img, pts_rect_depth
    

    def boxes3d_lidar_to_kitti_camera(self, boxes3d_lidar):
        """Convert 3D boxes from LiDAR to KITTI camera frame
        Args:
            boxes3d_lidar: (N, 7) [x, y, z, h, w, l, heading]
        Returns:
            boxes3d_camera: (N, 7) [x, y, z, h, w, l, ry] in rect camera coords
        """
        boxes3d_lidar_copy = copy.deepcopy(boxes3d_lidar)
        xyz_lidar = boxes3d_lidar_copy[:, 0:3] 
        h, w, l = boxes3d_lidar_copy[:, 3:4], boxes3d_lidar_copy[:, 4:5], boxes3d_lidar_copy[:, 5:6]
        heading = boxes3d_lidar_copy[:, 6:7]

        xyz_lidar[:, 2] -= h.reshape(-1) / 2
        xyz_cam = self.lidar_to_rect(xyz_lidar)
        r_y = -heading - np.pi / 2 # Adjust rotation (LiDAR-CW → Camera-CCW + axis correction)

        return np.concatenate([xyz_cam, h, w, l, r_y], axis=-1)
    

    def boxes3d_kitti_camera_to_imageboxes(self, boxes3d_camera, image_shape=None):
        """
        Args:
            boxes3d_camera: (N, 7) [x, y, z, h, w, l, ry] in rect camera coords
        Returns: 
            boxes_2d_preds: (N, 4) [x1, y1, x2, y2] = [xmin, ymin, xmax, ymax]
        """
        corners3d = boxes3d_to_corners3d_kitti_camera(boxes3d_camera)
        pts_img, _ = self.rect_to_img(corners3d.reshape(-1, 3))
        corners_in_image = pts_img.reshape(-1, 8, 2)

        min_uv = np.min(corners_in_image, axis=1)  # (N, 2)
        max_uv = np.max(corners_in_image, axis=1)  # (N, 2)
        boxes2d_image = np.concatenate([min_uv, max_uv], axis=1)
        if image_shape is not None:
            boxes2d_image[:, 0] = np.clip(boxes2d_image[:, 0], a_min=0, a_max=image_shape[1] - 1)
            boxes2d_image[:, 1] = np.clip(boxes2d_image[:, 1], a_min=0, a_max=image_shape[0] - 1)
            boxes2d_image[:, 2] = np.clip(boxes2d_image[:, 2], a_min=0, a_max=image_shape[1] - 1)
            boxes2d_image[:, 3] = np.clip(boxes2d_image[:, 3], a_min=0, a_max=image_shape[0] - 1)

        return boxes2d_image


    def parse_lidar_label(self, label_line):
        """Parse a line from LiDAR label file"""
        parts = label_line.strip().split()
        return {
            'type': str(parts[0]),
            'truncated': float(parts[1]),
            'occluded': int(parts[2]),
            'alpha': float(parts[3]),  # Convert to float
            'bbox': [float(parts[4]), (parts[5]), (parts[6]), (parts[7])],  # Convert to float
            'dimensions': [float(parts[8]), float(parts[9]), float(parts[10])],  # h, w, l
            'location': [float(parts[11]), float(parts[12]), float(parts[13])],   # x, y, z
            'rotation_z': float(parts[14]),  # Convert to float
            'score': float(parts[15])
        }


    def lidar_to_camera_label(self, lidar_label):
        """Convert LiDAR label to camera frame using OpenPCDet method"""
        # Extract box parameters
        x, y, z = lidar_label['location']
        h, w, l = lidar_label['dimensions']
        heading_lidar = lidar_label['rotation_z']
 
        # Create box array in OpenPCDet format [x,y,z,h,w,l,heding]
        box3d_lidar = np.array([[x, y, z, h, w, l, heading_lidar]])

        # Convert using OpenPCDet method
        box3d_camera = self.boxes3d_lidar_to_kitti_camera(box3d_lidar)
        x_rect, y_rect, z_rect, h, w, l, rotation_y = box3d_camera[0]

        box2d_camera = self.boxes3d_kitti_camera_to_imageboxes(box3d_camera, self.image_shape)
        xmin, ymin, xmax, ymax = box2d_camera[0]
        #print(f"{lidar_label['type']}")
        #print(f"Esti.: {xmin, ymin, xmax, ymax}\n")

        return {
            'type': lidar_label['type'],
            'truncated': float(lidar_label['truncated']),
            'occluded': int(lidar_label['occluded']),
            'alpha': lidar_label['alpha'],
            #'bbox': lidar_label['bbox'], # from gt data
            'bbox': [xmin, ymin, xmax, ymax], # calculated 2D boxes
            'dimensions': [h, w, l],  # h, w, l
            'location': [x_rect, y_rect, z_rect], # x, y, z
            'rotation_y': rotation_y,
            'score': float(lidar_label['score'])
        }
   
    
if __name__ == "__main__":

    single_file_mode = False

    # Initialize converter
    converter = LiDARtoCameraConverter()
    
    # Load dataset with calibration info
    dataset_path = Path("validation_pickle/kitti_val_dataset.pkl")
    converter.load_calib_from_pkl(dataset_path)

    if single_file_mode:
        test_label_path = "predictions/pred_bev_to_lidar_fp32/000002.txt"
        lidar_idx = Path(test_label_path).stem
        output_dir = "kitti_gt_annos/gt_lidar_to_camera_labels"

        try:
            # Get calibration data for this frame
            converter.get_calib_for_frame(lidar_idx)

            # Read and convert labels
            camera_labels = []
            with open(test_label_path, 'r') as f:
                for line in f:
                    #print("Input (LiDAR):", line.strip())
                    lidar_label = converter.parse_lidar_label(line)
                    camera_label = converter.lidar_to_camera_label(lidar_label)
                    camera_labels.append(camera_label)

                    output = f"{camera_label['type']} {camera_label['truncated']} {camera_label['occluded']} " \
                            f"{camera_label['alpha']} {camera_label['bbox']} " \
                            f"{camera_label['dimensions'][0]:.2f} {camera_label['dimensions'][1]:.2f} {camera_label['dimensions'][2]:.2f} " \
                            f"{camera_label['location'][0]:.2f} {camera_label['location'][1]:.2f} {camera_label['location'][2]:.2f} " \
                            f"{camera_label['rotation_y']:.2f} {camera_label['score']}"
                    #print("Output (Camera):", output)

                #save_transf_camera_labels(output_dir, lidar_idx, camera_labels)
                
        except FileNotFoundError:
            print(f"Label file not found: {test_label_path}")
        except ValueError as e:
            print(f"Error: {e}")

    else:
        import glob
        from progress.bar import IncrementalBar
        import os

        lidar_label_dir = "kitti_gt_annos_2/gt_bev_to_lidar_labels_2"
        output_dir = "kitti_gt_annos_2/gt_lidar_to_camera_labels_2"

        if not os.path.exists(lidar_label_dir):
            print(f"Directoy '{lidar_label_dir}' not found.")
            exit(1)
        
        lidar_label_files = glob.glob(os.path.join(lidar_label_dir, "*.txt"))
        bar = IncrementalBar('Processing', max=len(lidar_label_files), 
                             suffix='%(percent).1f%% - Estimated time: %(eta)ds')

        for lidar_label_file in lidar_label_files:
            lidar_idx = Path(lidar_label_file).stem
            converter.get_calib_for_frame(lidar_idx)

            camera_labels = []
            with open(lidar_label_file, 'r') as f:
                for line in f:
                    lidar_label = converter.parse_lidar_label(line)
                    camera_label = converter.lidar_to_camera_label(lidar_label)
                    camera_labels.append(camera_label)

                #save_transf_camera_labels(output_dir, lidar_idx, camera_labels)

            bar.next()
            
        bar.finish()

## pred_lidar2camera_frame test

In [ ]:
import numpy as np
import pickle
from pathlib import Path
import copy
from vod.label_transformation.utils.utils import save_transf_camera_labels, cart_to_hom, boxes3d_to_corners3d_kitti_camera

class PredLiDARtoCameraConverter:
    def __init__(self):
        """
        Initialize converter with calibration data from dataset.
        """
        self.dataset = None
        self.calib_data = {}
        self.P2 = None
        self.R0 = None
        self.V2C = None
        self.image_shape = None

   
    def load_calib_from_pkl(self, dataset_path):
        """
        Load dataset to get calibration data from .pkl file.

        Args:
            dataset_path (str or Path): Path to the pickle file 

        Returns:
            None
        """
        with open(dataset_path, 'rb') as f:
            self.dataset = pickle.load(f)
            
        # Extract calibration data for each frame
        for frame in self.dataset:
            if 'point_cloud' in frame and 'calib' in frame:
                lidar_idx = frame['point_cloud']['lidar_idx']
                calib = frame['calib']

                if 'image' in frame:
                    image_shape = frame['image']['image_shape']

                self.calib_data[lidar_idx] = {
                    'P2': calib['P2'][:3],  # 3 x 4
                    'R0': calib['R0_rect'][:3, :3],  # 3 x 3
                    'Tr_velo2cam': calib['Tr_velo_to_cam'][:3],  # 3 x 4
                    'image_shape': image_shape # [height, width]    
                }
    

    def get_calib_for_frame(self, lidar_idx):
        """
        Get calibration data for specific frame.
        
        Args:
            lidar_idx (str): LiDAR frame identifier/index

        Returns:
            dict: Calibration data dictionary containing P2, R0, and Tr_velo2cam matrices
        """
        if lidar_idx not in self.calib_data:
            raise ValueError(f"No calibration data found for frame {lidar_idx}")
        
        calib = self.calib_data[lidar_idx]
        self.P2 = calib['P2']
        self.R0 = calib['R0']
        self.V2C = calib['Tr_velo2cam']
        self.image_shape = calib['image_shape']

        return calib
    

    def lidar_to_rect(self, pts_lidar):
        """
        Convert points from LiDAR to camera rect coordinates.

        Args:
            pts_lidar: Points in LiDAR coordinates, shape (N, 3)
            
        Returns:
            np.ndarray: Points in camera rectified coordinates, shape (N, 3)   
        """
        pts_lidar_hom = cart_to_hom(pts_lidar)
        pts_rect = np.dot(pts_lidar_hom, np.dot(self.V2C.T, self.R0.T))

        return pts_rect
    

    def rect_to_img(self, pts_rect):
        """
        Project rectified camera coordinates to image plane.
        
        Args:
            pts_rect: Points in rectified camera coordinates, shape (N, 3)
            
        Returns:
            tuple: (pts_img, pts_rect_depth)
                - pts_img (np.ndarray): Image coordinates, shape (N, 2)
                - pts_rect_depth (np.ndarray): Depth values in rectified camera coords, shape (N,)
        """
        pts_rect_hom = cart_to_hom(pts_rect)
        pts_2d_hom = np.dot(pts_rect_hom, self.P2.T)
        pts_img = (pts_2d_hom[:, 0:2].T / pts_rect_hom[:, 2]).T  # (N, 2)
        pts_rect_depth = pts_2d_hom[:, 2] - self.P2.T[3, 2]  # depth in rect camera coord

        return pts_img, pts_rect_depth


    def boxes3d_lidar_to_kitti_camera_pred(self, boxes3d_lidar):
        """
        Convert 3D boxes from LiDAR to KITTI camera frame.

        Args:
            boxes3d_lidar (np.ndarray): 3D boxes in LiDAR coordinates, shape (N, 7)
                                       Format: [x, y, z, h, w, l, heading]
                                       
        Returns:
            np.ndarray: 3D boxes in camera coordinates, shape (N, 7)
                       Format: [x, y, z, h, w, l, rotation_y] in rectified camera coords
        """
        boxes3d_lidar_copy = copy.deepcopy(boxes3d_lidar)
        xyz_lidar = boxes3d_lidar_copy[:, 0:3] 
        h, w, l = boxes3d_lidar_copy[:, 3:4], boxes3d_lidar_copy[:, 4:5], boxes3d_lidar_copy[:, 5:6]
        heading = boxes3d_lidar_copy[:, 6:7]

        xyz_lidar[:, 2] -= h.reshape(-1) / 2
        xyz_cam = self.lidar_to_rect(xyz_lidar)
        # Turn the rotation direction from CW (LiDAR) to CCW (Camera). 
        # Shift the reference angle, as the y-axis is vertical in the 
        # camera frame and the z-axis in the LiDAR frame.
        ry = -heading - np.pi / 2 

        return np.concatenate([xyz_cam, h, w, l, ry], axis=-1)
    

    def boxes3d_kitti_camera_to_imageboxes(self, boxes3d_camera, image_shape=None):
        """
        Convert 3D camera boxes to 2D image bounding boxes.
        
        Args:
            boxes3d_camera (np.ndarray): 3D boxes in camera coordinates, shape (N, 7)
                                        Format: [x, y, z, h, w, l, rotation_y]
            image_shape (list, optional): Image dimensions [height, width] for clipping.
                                         If None, no clipping is applied
                                         
        Returns:
            np.ndarray: 2D bounding boxes, shape (N, 4)
                       Format: [xmin, ymin, xmax, ymax] in image coordinates
        """
        corners3d = boxes3d_to_corners3d_kitti_camera(boxes3d_camera)
        pts_img, _ = self.rect_to_img(corners3d.reshape(-1, 3))
        corners_in_image = pts_img.reshape(-1, 8, 2)

        min_uv = np.min(corners_in_image, axis=1)  # (N, 2)
        max_uv = np.max(corners_in_image, axis=1)  # (N, 2)
        boxes2d_image = np.concatenate([min_uv, max_uv], axis=1)
        if image_shape is not None:
            boxes2d_image[:, 0] = np.clip(boxes2d_image[:, 0], a_min=0, a_max=image_shape[1] - 1)
            boxes2d_image[:, 1] = np.clip(boxes2d_image[:, 1], a_min=0, a_max=image_shape[0] - 1)
            boxes2d_image[:, 2] = np.clip(boxes2d_image[:, 2], a_min=0, a_max=image_shape[1] - 1)
            boxes2d_image[:, 3] = np.clip(boxes2d_image[:, 3], a_min=0, a_max=image_shape[0] - 1)

        return boxes2d_image

    
    def parse_lidar_label(self, label_line):
        """
        Parse a single line from LiDAR label file into structured format.
        
        Args:
            label_line (str): Single line from label file containing space-separated values
                             Format: type truncated occluded alpha bbox dimensions location rotation_z score
                             
        Returns:
            dict: Parsed label data with keys
        """
        parts = label_line.strip().split()
        return {
            'type': str(parts[0]),
            'truncated': float(parts[1]),
            'occluded': int(parts[2]),
            'alpha': float(parts[3]), 
            'bbox': [float(parts[4]), float(parts[5]), float(parts[6]), float(parts[7])],  # xmin, ymin, xmax, ymax
            'dimensions': [float(parts[8]), float(parts[9]), float(parts[10])],  # h, w, l
            'location': [float(parts[11]), float(parts[12]), float(parts[13])],   # x, y, z
            'rotation_z': float(parts[14]), 
            'score': float(parts[15])
        }

    def convert_label(self, lidar_label):
        """
        Convert LiDAR label to camera frame using OpenPCDet transformation method.
        
        Args:
            lidar_label (dict): Label data in LiDAR frame format from parse_lidar_label()
            
        Returns:
            dict: Label data converted to camera frame format with keys
        """
        # Extract box parameters
        x, y, z = lidar_label['location']
        h, w, l = lidar_label['dimensions']
        heading = lidar_label['rotation_z']
 
        # Create box array in OpenPCDet format [x,y,z,h,w,l,r]
        box3d_lidar = np.array([[x, y, z, h, w, l, heading]])

        # Convert LiDAR 3D Boxes to Camera 3D Boxes using OpenPCDet method
        box3d_camera = self.boxes3d_lidar_to_kitti_camera_pred(box3d_lidar)
        x_rect, y_rect, z_rect, h, w, l, rotation_y = box3d_camera[0]
        
        # Convert Camera 3D Boxes to Camera 2D Boxes using OpenPCDet method
        box2d_camera = self.boxes3d_kitti_camera_to_imageboxes(box3d_camera, self.image_shape)
        xmin, ymin, xmax, ymax = box2d_camera[0]

        # Create Camera label in default KITTI format
        return {
            'type': lidar_label['type'],
            'truncated': float(lidar_label['truncated']), # adopted dummy value
            'occluded': int(lidar_label['occluded']), # adopted dummy value
            'alpha': lidar_label['alpha'], # adopted dummy value
            'bbox': [xmin, ymin, xmax, ymax], # Use calculated 2D bbox values
            'dimensions': [h, w, l],  # h, w, l 
            'location': [x_rect, y_rect, z_rect], # x, y, z
            'rotation_y': rotation_y,
            'score': float(lidar_label['score'])
        }
    
if __name__ == "__main__":

    single_file_mode = False

    # Initialize converter
    converter = PredLiDARtoCameraConverter()

    # Load dataset with calibration info
    dataset_path = Path("validation_pickle/kitti_val_dataset.pkl")
    converter.load_calib_from_pkl(dataset_path)


    if single_file_mode:
        pred_file = "predictions/all_bev_preds_minAreaRect()/pred_bev_to_lidar_fp32/000006.txt"
        lidar_idx = Path(pred_file).stem
        output_dir = "predictions/all_bev_preds_minAreaRect()/pred_lidar_to_camera_fp32"

        try:
            # Get calibration data for this frame
            converter.get_calib_for_frame(lidar_idx)

            # Read and convert predictions
            camera_labels = []
            with open(pred_file, 'r') as f:
                for line in f:
                    lidar_label = converter.parse_lidar_label(line)
                    camera_label = converter.convert_label(lidar_label)
                    camera_labels.append(camera_label)

                    output = f"{camera_label['type']} {camera_label['truncated']} {camera_label['occluded']} " \
                            f"{camera_label['alpha']} {camera_label['bbox']} " \
                            f"{camera_label['dimensions'][0]:.2f} {camera_label['dimensions'][1]:.2f} {camera_label['dimensions'][2]:.2f} " \
                            f"{camera_label['location'][0]:.2f} {camera_label['location'][1]:.2f} {camera_label['location'][2]:.2f} " \
                            f"{camera_label['rotation_y']:.2f} {camera_label['score']}"
                    #print("Output (Camera):", output)

                #save_transf_camera_labels(output_dir, lidar_idx, camera_labels)
        
        except FileNotFoundError:
            print(f"Label file not found: {pred_file}")
        except ValueError as e:
            print(f"Error: {e}")
    
    else:
        import os
        import glob
        from progress.bar import IncrementalBar

        # Model: FP32, Yaw range: [-pi/4...3pi/4]
        pred_dir = "predictions/all_bev_preds_minAreaRect()/pred_bev_to_lidar_fp32"
        # Model: FP16, Yaw range: [-pi/4...3pi/4]
        #pred_dir = ""
        # Model: INT8, Yaw range: [-pi/4...3pi/4]
        #pred_dir = ""

        # Model: FP32, Yaw range: [0, pi/2]
        #pred_dir = "predictions/all_bev_preds_regularized/pred_bev_to_lidar_fp32_rgd"
        # Model: FP16, Yaw range: [0, pi/2]
        #pred_dir = ""
        # Model: INT8, Yaw range: [0, pi/2]
        #pred_dir = ""

        output_dir = "predictions/all_bev_preds_minAreaRect()/pred_lidar_to_camera_fp32"

        if not os.path.exists(pred_dir):
            print(f"Directoy '{pred_dir}' not found.")
            exit(1)
        
        lidar_label_files = glob.glob(os.path.join(pred_dir, "*.txt"))
        bar = IncrementalBar('Processing', max=len(lidar_label_files), 
                             suffix='%(percent).1f%% - Estimated time: %(eta)ds')

        for lidar_label_file in lidar_label_files:
            lidar_idx = Path(lidar_label_file).stem
            converter.get_calib_for_frame(lidar_idx)

            camera_labels = []
            with open(lidar_label_file, 'r') as f:
                for line in f:
                    lidar_label = converter.parse_lidar_label(line)
                    camera_label = converter.convert_label(lidar_label)
                    camera_labels.append(camera_label)

                save_transf_camera_labels(output_dir, lidar_idx, camera_labels)

            bar.next()
            
        bar.finish()